# Context Parallelism: Hands-On Tutorial

**Scale to 128K+ Token Sequences with Distributed Attention**

---

## The Problem

Attention memory scales as **O(S²)** where S is sequence length:

| Sequence Length | Attention Matrix | Memory per Layer |
|-----------------|------------------|------------------|
| 1K | 1K × 1K | 64 MB |
| 4K | 4K × 4K | 1 GB |
| 16K | 16K × 16K | 16 GB |
| 64K | 64K × 64K | 256 GB ← Impossible! |

## The Solution: Context Parallelism

Split queries across GPUs. Each GPU computes attention for only S/CP queries:

```
Without CP:  Each GPU computes (S × S) attention     → O(S²)
With CP=2:   Each GPU computes (S/2 × S) attention   → O(S²/2) = 2x smaller!
With CP=4:   Each GPU computes (S/4 × S) attention   → O(S²/4) = 4x smaller!
```

---

## Step 0: Check GPU Setup

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader
import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')

## Step 1: Verify NCCL Works

First, let's make sure distributed communication is working on your pod.

In [ ]:
%%writefile step1_test_nccl.py
"""Step 1: Verify NCCL works on fresh pod"""
import torch
import torch.distributed as dist

dist.init_process_group('nccl')
rank = dist.get_rank()
ws = dist.get_world_size()
torch.cuda.set_device(rank)

print(f'[GPU {rank}] Initialized', flush=True)

x = torch.tensor([rank + 1.0], device=f'cuda:{rank}')
dist.all_reduce(x)

print(f'[GPU {rank}] all_reduce result: {x.item()} (expected: {ws * (ws + 1) / 2})', flush=True)

if rank == 0:
    print('\n✓ NCCL is working! Proceed to step 2.\n', flush=True)

dist.destroy_process_group()

In [ ]:
!torchrun --nproc_per_node=2 step1_test_nccl.py

**Expected output:**
```
[GPU 0] Initialized
[GPU 1] Initialized
[GPU 0] all_reduce result: 3.0 (expected: 3.0)
[GPU 1] all_reduce result: 3.0 (expected: 3.0)

✓ NCCL is working! Proceed to step 2.
```

---

## Step 2: Run Context Parallelism Comparison

This compares:
1. **No CP**: Full (S × S) attention matrix on each GPU
2. **With CP**: Smaller (S/2 × S) attention matrix on each GPU

In [ ]:
%%writefile step2_cp_comparison.py
"""Step 2: Context Parallelism Comparison"""
import torch
import torch.nn.functional as F
import torch.distributed as dist
import time
import math
import json

def main():
    dist.init_process_group('nccl')
    rank = dist.get_rank()
    ws = dist.get_world_size()
    torch.cuda.set_device(rank)
    device = f'cuda:{rank}'
    
    B, H, D = 4, 16, 64  # batch, heads, head_dim
    
    if rank == 0:
        print('\n' + '=' * 65)
        print('  CONTEXT PARALLELISM COMPARISON')
        print('=' * 65)
        print(f'  GPUs: {ws}')
        print(f'  Batch: {B}, Heads: {H}, Head dim: {D}')
        print('=' * 65 + '\n')
    
    all_results = {}
    
    for S in [1024, 2048, 4096]:
        S_local = S // ws
        
        if rank == 0:
            print(f'--- Sequence Length: {S} ---')
        
        # TEST 1: Standard Attention (No CP)
        Q = torch.randn(B, H, S, D, device=device)
        K = torch.randn(B, H, S, D, device=device)
        V = torch.randn(B, H, S, D, device=device)
        
        for _ in range(3):
            attn = (Q @ K.transpose(-2, -1)) / math.sqrt(D)
            out = F.softmax(attn, dim=-1) @ V
        torch.cuda.synchronize()
        
        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        for _ in range(10):
            attn = (Q @ K.transpose(-2, -1)) / math.sqrt(D)
            out = F.softmax(attn, dim=-1) @ V
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        
        mem_no_cp = torch.cuda.max_memory_allocated() / 1024**2
        time_no_cp = (t1 - t0) / 10 * 1000
        
        del Q, K, V, attn, out
        torch.cuda.empty_cache()
        
        # TEST 2: CP-Style Attention
        Q_local = torch.randn(B, H, S_local, D, device=device)
        K_full = torch.randn(B, H, S, D, device=device)
        V_full = torch.randn(B, H, S, D, device=device)
        
        for _ in range(3):
            attn = (Q_local @ K_full.transpose(-2, -1)) / math.sqrt(D)
            out = F.softmax(attn, dim=-1) @ V_full
        torch.cuda.synchronize()
        
        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        for _ in range(10):
            attn = (Q_local @ K_full.transpose(-2, -1)) / math.sqrt(D)
            out = F.softmax(attn, dim=-1) @ V_full
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        
        mem_cp = torch.cuda.max_memory_allocated() / 1024**2
        time_cp = (t1 - t0) / 10 * 1000
        
        del Q_local, K_full, V_full, attn, out
        torch.cuda.empty_cache()
        
        if rank == 0:
            reduction = (mem_no_cp - mem_cp) / mem_no_cp * 100
            print(f'  No CP:   attn=({S}x{S})       mem={mem_no_cp:7.1f} MB  time={time_no_cp:6.2f} ms')
            print(f'  CP={ws}:    attn=({S_local}x{S})     mem={mem_cp:7.1f} MB  time={time_cp:6.2f} ms')
            print(f'  Reduction: {reduction:.1f}%\n')
            
            all_results[str(S)] = {
                'no_cp': {'mem_mb': round(mem_no_cp, 1), 'time_ms': round(time_no_cp, 2)},
                'with_cp': {'mem_mb': round(mem_cp, 1), 'time_ms': round(time_cp, 2)},
                'reduction_pct': round(reduction, 1)
            }
    
    if rank == 0:
        with open('cp_results.json', 'w') as f:
            json.dump(all_results, f, indent=2)
        
        print('=' * 65)
        print('  SUMMARY')
        print('=' * 65)
        print(f"  {'Seq':<8} {'No CP Mem':>12} {'CP Mem':>12} {'Reduction':>12}")
        print('  ' + '-' * 44)
        for s, r in all_results.items():
            print(f"  {s:<8} {r['no_cp']['mem_mb']:>10.1f} MB {r['with_cp']['mem_mb']:>10.1f} MB {r['reduction_pct']:>10.1f}%")
        print('=' * 65)
        print('  Results saved to cp_results.json')
        print('=' * 65 + '\n')
    
    dist.destroy_process_group()

if __name__ == '__main__':
    main()

In [ ]:
!torchrun --nproc_per_node=2 step2_cp_comparison.py

**Expected output:**
```
=================================================================
  CONTEXT PARALLELISM COMPARISON
=================================================================
  GPUs: 2

--- Sequence Length: 1024 ---
  No CP:   attn=(1024x1024)     mem=  XXX.X MB  time=  X.XX ms
  CP=2:    attn=(512x1024)      mem=  XXX.X MB  time=  X.XX ms
  Reduction: ~40-50%

--- Sequence Length: 2048 ---
  ...
```

---

## Step 3: Generate Beautiful Plots

In [ ]:
!pip install matplotlib -q

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

# Load results
with open('cp_results.json') as f:
    results = json.load(f)

seq_lens = [int(s) for s in results.keys()]
no_cp_mem = [results[str(s)]['no_cp']['mem_mb'] for s in seq_lens]
cp_mem = [results[str(s)]['with_cp']['mem_mb'] for s in seq_lens]
reductions = [results[str(s)]['reduction_pct'] for s in seq_lens]

# Create figure
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Context Parallelism: Performance Analysis', fontsize=14, fontweight='bold')

colors = {'no_cp': '#E74C3C', 'cp': '#2ECC71', 'blue': '#3498DB'}

# Plot 1: Memory Comparison
ax1 = axes[0]
x = np.arange(len(seq_lens))
width = 0.35
bars1 = ax1.bar(x - width/2, no_cp_mem, width, label='No CP', color=colors['no_cp'], edgecolor='black')
bars2 = ax1.bar(x + width/2, cp_mem, width, label='With CP', color=colors['cp'], edgecolor='black')
ax1.set_xlabel('Sequence Length')
ax1.set_ylabel('Peak Memory (MB)')
ax1.set_title('Memory Usage')
ax1.set_xticks(x)
ax1.set_xticklabels(seq_lens)
ax1.legend()
for bar in bars1:
    ax1.annotate(f'{bar.get_height():.0f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)
for bar in bars2:
    ax1.annotate(f'{bar.get_height():.0f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

# Plot 2: Memory Reduction %
ax2 = axes[1]
bars = ax2.bar([str(s) for s in seq_lens], reductions, color=colors['blue'], edgecolor='black')
ax2.set_xlabel('Sequence Length')
ax2.set_ylabel('Memory Reduction (%)')
ax2.set_title('Memory Savings with CP')
ax2.set_ylim(0, max(reductions) * 1.3)
for bar, val in zip(bars, reductions):
    ax2.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 xytext=(0, 3), textcoords='offset points', ha='center', fontweight='bold')

# Plot 3: Theoretical Scaling
ax3 = axes[2]
theo_seq = [512, 1024, 2048, 4096, 8192, 16384, 32768]
B, H = 4, 16
for cp_deg, color, style in [(1, colors['no_cp'], '-'), (2, colors['cp'], '--'), (4, '#9B59B6', ':')]:
    mem = [B * H * (s/cp_deg) * s * 4 / 1024**2 for s in theo_seq]
    label = f'CP={cp_deg}' if cp_deg > 1 else 'No CP'
    ax3.plot(theo_seq, mem, color=color, linestyle=style, linewidth=2, marker='o', markersize=5, label=label)
ax3.axhline(y=40000, color='red', linestyle='--', alpha=0.7)
ax3.text(theo_seq[-1]*0.5, 42000, '40GB GPU Limit', fontsize=9, color='red')
ax3.set_xlabel('Sequence Length')
ax3.set_ylabel('Attention Memory (MB)')
ax3.set_title('Memory Scaling (Theoretical)')
ax3.set_xscale('log', base=2)
ax3.set_yscale('log')
ax3.legend(loc='upper left')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cp_comparison.png', dpi=150, bbox_inches='tight', facecolor='white')
print('✓ Saved: cp_comparison.png')
plt.show()

---

## Key Takeaways

### Attention Memory Scaling

| Approach | Attention Matrix | Memory | When to Use |
|----------|------------------|--------|-------------|
| No CP | (S × S) | O(S²) | Short sequences (<4K) |
| CP=2 | (S/2 × S) | O(S²/2) | Medium sequences (4K-16K) |
| CP=4 | (S/4 × S) | O(S²/4) | Long sequences (16K-64K) |
| CP=8 | (S/8 × S) | O(S²/8) | Very long sequences (64K+) |

### How Context Parallelism Works

```
Standard Attention (each GPU):
  Q: (B, H, S, D)
  K: (B, H, S, D)  
  Attention = Q @ K.T = (B, H, S, S)  ← Full S×S matrix!

Context Parallel Attention (each GPU with CP=2):
  Q_local: (B, H, S/2, D)   ← Each GPU has half the queries
  K_full:  (B, H, S, D)     ← All keys (gathered from all GPUs)
  Attention = Q_local @ K_full.T = (B, H, S/2, S)  ← Half the size!
```

### The Complete Parallelism Stack

```
For training LLaMA-70B with 128K context:

  TP=8 + SP=8   (within node, NVLink)
      ↓
  CP=4          (for long sequences)
      ↓  
  PP=4          (across nodes, for huge models)
      ↓
  DP=8          (across replicas, for throughput)

  Total: 8 × 4 × 4 × 8 = 1024 GPUs
```

In [ ]:
print('Tutorial complete!')
print('\nFiles generated:')
!ls -la *.json *.png 2>/dev/null || echo '  (run steps above to generate files)'